# Implementation homework: fictional library FAQ

Adaptation of the Lecture 02 LoRA notebook. Keep the original model and 20-epoch training configuration, and change the topic to five fictional library rules. No results have been generated yet.

Run sections 0?7 in a fresh Colab T4 runtime. Section 8 is optional. Results are written to `/content/library_lora_lab`.

The dataset contains 14 training, 7 validation, and 21 test examples. The 15 grounded test questions paraphrase the five rules. The 6 ungrounded test questions ask about omitted policies, including related situations that cannot be inferred from the rules. Desired output remains `{"answer": string, "needs_review": boolean}`.

Evaluation uses the original keyword-based scorer; inspect answers manually as well as reporting the metrics. Training examples and test paraphrases concern the same rules, so this evaluates behavior on this FAQ rather than general knowledge.


## 0. Check the Environment and Install Libraries

**What to Do**: Check that a GPU is allocated and install libraries pinned to verified versions.

**What to Look For**

- The first cell displays `Tesla T4`.
- The second cell ends with a list of versions (such as `transformers: 5.18.0`). Installation takes about 1 minute.

**Troubleshooting**

- "No GPU found": select **Runtime → Change runtime type → T4 GPU** and run the cell again. If demand prevents allocation, try again later.
- Red pip `ERROR: pip's dependency resolver ...` messages (about `cudf` or `pyarrow`) warn of conflicts with other packages preinstalled in Colab. They do not affect this exercise, so continue.

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU runtimes do not have nvidia-smi
    gpu_info = ""
print(gpu_info or "No GPU found. Select Runtime → Change runtime type → T4 GPU.")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Versions verified on a Colab T4 (Python 3.13, torch 2.11.0+cu130). Use the torch bundled with Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 requires transformers 5.x
    "peft": "0.21.1",           # LoRA implementation
    "trl": "1.14.2",            # Training loop (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # The bundled torchao (0.10) is incompatible with peft 0.21 and causes ImportError during training. Remove it since it is unused
%pip install -q {spec}

# Dependency warnings about cudf-cu13 and pyarrow appear as ERROR, but can be ignored because cudf is unused
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. Settings

**What to Do**: Collect the model name, number of training epochs (passes through the data), and other settings in a single dictionary, `CONFIG`. This dictionary is recorded as is in the submission file in section 7 so you can later identify "the settings used for each experiment".

- Fixing `revision` ensures the same weights are used whenever you run the notebook. This keeps conditions consistent for before/after and cross-person comparisons.
- `num_train_epochs: 20` means 20 passes through the training data (14 examples). With 2 steps per pass, this totals 40 steps. The original Japanese run took 4–5 minutes on a T4; timing may differ with the translated data.
- `gen: {"do_sample": False}` uses greedy generation. This fixes generation conditions before and after training and removes the effect of sampling variation.

Setting `USE_DRIVE` to `True` mounts Google Drive and saves the trained adapter and result files to Drive as well. Use this to retain results if the session disconnects. Leave it `False` if you will finish in one session.

**What to Look For**: The settings are displayed as JSON.

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # Small instruction-tuned model (Apache-2.0). Fits on a T4 without quantization
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # Pinned to use the same weights on every run
    "seed": 42,
    "num_train_epochs": 20,  # Number of passes over training data (14 examples). 2 steps per pass gives 40 steps (4–5 minutes on a T4 in the original Japanese run)
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA matrix dimension (smaller means fewer trainable parameters)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # Fix evaluation to greedy generation (consistent before/after conditions)
    "work_dir": "/content/library_lora_lab",                          # Result file location (visible in the file list on the left)
    "drive_dir": "/content/drive/MyDrive/aie2026/library_lora_lab",  # Save location when USE_DRIVE=True
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # Set True to mount Google Drive and save adapters and outputs
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/library_lora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/library_lora_lab"
}


## 2. Generate the fictional library FAQ and data

Five rules cover loan duration, book limit, return location, opening hours, and renewal. Each rule has two training templates, one validation template, and three test templates. Unknown-policy questions are split into four training, two validation, and six test examples.

Unknown questions deliberately omit facts such as fines, membership, reservations, and exceptions. A general rule does not establish how an exception is handled. Expected output for these questions requests confirmation with `needs_review=true`.

Check `train=14 valid=7 test=21 gold=21`.


In [4]:
# Fictional library policy: these rules describe no real institution.
FAQ = {'loan_period': 'Books may be borrowed for 14 days.',
 'book_limit': 'Each member may borrow up to 3 books at a time.',
 'return_location': 'Return borrowed books at the front desk.',
 'opening_hours': 'The library is open Monday to Friday from 09:00 to 17:00.',
 'renewal': 'Each book may be renewed once through the library portal.'}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

GROUNDED = {'loan_period': (['How long can I borrow a book?',
                  'What is the loan period for books?',
                  'For how many days may I keep a borrowed book?',
                  'When must I return a book after borrowing it?'],
                 ['14']),
 'book_limit': (['How many books can I borrow at once?',
                 'What is the borrowing limit per member?',
                 'What is the maximum number of books I may borrow?',
                 'How many books may a member have on loan simultaneously?'],
                ['3']),
 'return_location': (['Where should I return borrowed books?',
                      'Where do I take books when returning them?',
                      'What is the book return location?',
                      'Where can I hand back a borrowed book?'],
                     ['front desk']),
 'opening_hours': (['When is the library open?',
                    'What are the library opening hours?',
                    'Tell me the library opening days and times.',
                    'On which days and at what times can I visit the library?'],
                   ['Monday', 'Friday', '09:00', '17:00']),
 'renewal': (['How do I renew a book and how many renewals are allowed?',
              'Where can I renew a book, and how often?',
              'Tell me the renewal method and limit.',
              'How many times may I renew a book and through which service?'],
             ['once', 'library portal'])}

# These questions concern policies deliberately absent from the FAQ.
UNGROUNDED = ['Are there fines for overdue books?',
 'Can visitors without membership borrow books?',
 'Can I reserve a study room?',
 'Is Wi-Fi available?',
 'Can I borrow electronic readers?',
 'Are lost books replaced for free?',
 'Can I print documents at the library?',
 'Is there a membership fee?']
TEST_EXTRA_GROUNDED = {'loan_period': ['How many days does a book loan last?',
                 'What is the permitted borrowing duration for a book?'],
 'book_limit': ['What is the simultaneous book-loan limit?',
                'What is the largest number of books a member can check out at once?'],
 'return_location': ['Which desk accepts book returns?',
                     'Where should a member bring books at the end of a loan?'],
 'opening_hours': ['What is the weekday opening schedule?',
                   'Give the opening days and the daily opening and closing times.'],
 'renewal': ['Which service handles renewals and what is the renewal limit?',
             'How often can a borrowed book be renewed, and where is this done?']}
TEST_EXTRA_UNGROUNDED = ['Can I return books after closing time?',
 'Can I renew a book that another member has reserved?',
 'Are children allowed to become members?',
 'Can I pay overdue fines online?']
UNGROUNDED_ANSWER = "This FAQ does not contain that information. Confirmation from the library staff is needed."

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # Fix the first 2 templates to train, the third to valid, and the fourth to test (keep paraphrases separate)
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # Assign each group of 4 questions without evidence as train 2 / valid 1 / test 1
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["Confirmation"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["Confirmation"]}

# train has 14 examples (10 with evidence, 4 without). Set repetitions with num_train_epochs in section 4 instead of duplicating data
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "Questions are duplicated across train / valid / test. Review the templates"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("One training example | Question:", train[0]["question"], "| Desired output:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

train=14 valid=7 test=21 gold=21
One training example | Question: How long can I borrow a book? | Desired output: {"answer": "Books may be borrowed for 14 days.", "needs_review": false}


## 3. Load the Model and Evaluate Before Training

**What to Do**: Load the base model without quantization (32-bit), have it answer the 21 test questions, and score the results.

The prompt (`SYSTEM`) says "Always return `answer` and `needs_review` in one JSON line". The **format has already been instructed**. Before training, check whether the model follows this instruction.

**Scoring** (the same functions are used in section 5)

- **Format**: The output parses as JSON, has exactly two keys (`answer` and `needs_review`), and uses string and boolean types respectively.
- **Content**: `needs_review` matches the expected answer, and `answer` contains the required keywords (for example, "14" for the loan duration).

Content scoring requires a matching `needs_review` value. Outputs with a missing key therefore receive `Fail` for content even if the wording is correct. Check the wording yourself in the table. The format-failure codes mean: `json_error` = JSON parsing failed; `not_object` = the JSON value is not an object; `extra_keys` = unexpected keys are present; `missing_key` = a required key is absent; `wrong_type` = an answer or review flag has the wrong type.

`Qwen/Qwen3.5-0.8B` also supports images, but loading it with `AutoModelForCausalLM` uses only the text component.

**What to Look For**

- First cell: a message such as `Weight memory (GB): 3.01`. Downloading the model (about 1.75GB) takes about 1 minute.
- Second cell: a one-line summary (`format_rate`, `content_rate`, etc.) and a table of outputs for 21 questions. Evaluation takes about 1 minute. In the original Japanese run, many before-training outputs were missing `needs_review`; check which keys are missing in your run.

**Harmless Warnings**

- `You are sending unauthenticated requests to the HF Hub`: this means the model is being downloaded without signing in to Hugging Face. Public models can be downloaded without signing in. Signing in is needed in the following cases, regardless of model size:
  - Using a model that requires agreement to terms (a gated model, such as Llama or Gemma) or a private repository.
  - Uploading your own models or data to the Hub.
  - Sending many requests in a short time and hitting a limit. Without signing in, the limit is 3,000 file retrievals per IP address per 5 minutes (published figure as of September 2025; [Hub rate limits](https://huggingface.co/docs/hub/rate-limits)). Signing in raises it.

  This notebook's model is public and requires no agreement. Loading it makes at most a few dozen requests, well below the limit. If `429 Too Many Requests` does occur, the library reads the waiting time and retries automatically. If signing in becomes necessary, register a Hugging Face token named `HF_TOKEN` using the key icon (Secrets) on Colab's left sidebar and allow this notebook to access it; it will be used automatically.
- `... is falling back to its reference PyTorch implementation ...`: a fast implementation for some of this model's layers (linear attention) is not installed, so a slower but correct implementation is used. About 4 lines appear.

**Read Before Continuing**: Check the table for "which keys are missing" and "whether the wording agrees with the FAQ". In section 5, compare this table alongside outputs after training. Original outputs are also saved in `/content/library_lora_lab/eval_before.json` (open it from the file list on the left).

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# No quantization. Keep weights in 32-bit (specify dtype because omitting it loads the distributed precision)
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| Weight memory (GB):", BASE_MEMORY_GB)

# The prompt instructs the format. Before training, check whether the model follows it
SYSTEM = "You answer library FAQ questions. Use only the provided FAQ as evidence and always answer in one JSON line. Format: {\"answer\": string, \"needs_review\": boolean}. For information absent from the FAQ, write 'Confirmation is needed' in answer and set needs_review to true."

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\nQuestion: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: leave the thinking block empty and generate only the answer (explicit for consistent conditions across models)
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | Weight memory (GB): 3.01


In [6]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # Display original outputs without truncation

def check_format(text: str):
    '''Check JSON parsing, required keys, and types. Also return the reason for failure'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''Content passes if needs_review matches the expected answer and answer contains required keywords'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "Pass" if ok else "Fail"

def evaluate(model, label: str):
    '''Generate and score the 21 test questions, returning a summary and per-question results. Save original outputs in eval_<label>.json'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # If the format is broken but a JSON-like portion can be parsed, score content separately
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# Per-question outputs. Compare with expected needs_review and check for missing keys and agreement with FAQ wording
display(pd.DataFrame([{"Question": r["question"], "Expected needs_review": r["gold_needs_review"], "Before-training output": r["raw"],
                       "Format": mark(r["format_ok"]), "Format failure reason": r["format_reason"], "Content": mark(r["content_ok"])} for r in before_rows]))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


{"label": "before", "n": 21, "format_rate": 1.0, "content_rate": 0.42857142857142855, "both_rate": 0.42857142857142855, "seconds": 43.4}


,Question,Expected needs_review,Before-training output,Format,Format failure reason,Content
0,When must I return a book after borrowing it?,False,"{""answer"": ""Return borrowed books at the front desk."", ""needs_review"": true}",Pass,,Fail
1,How many days does a book loan last?,False,"{""answer"": ""14"", ""needs_review"": false}",Pass,,Pass
2,What is the permitted borrowing duration for a book?,False,"{""answer"": ""14 days"", ""needs_review"": false}",Pass,,Pass
3,How many books may a member have on loan simultaneously?,False,"{""answer"": ""3"", ""needs_review"": false}",Pass,,Pass
4,What is the simultaneous book-loan limit?,False,"{""answer"": ""3"", ""needs_review"": false}",Pass,,Pass
5,What is the largest number of books a member can check out at once?,False,"{""answer"": ""3"", ""needs_review"": false}",Pass,,Pass
6,Where can I hand back a borrowed book?,False,"{""answer"": ""The front desk"", ""needs_review"": true}",Pass,,Fail
7,Which desk accepts book returns?,False,"{""answer"": ""Front desk"", ""needs_review"": false}",Pass,,Fail
8,Where should a member bring books at the end of a loan?,False,"{""answer"": ""Return borrowed books at the front desk."", ""needs_review"": true}",Pass,,Fail
9,On which days and at what times can I visit the library?,False,"{""answer"": ""Monday to Friday from 09:00 to 17:00"", ""needs_review"": false}",Pass,,Pass


## 4. LoRA Training (Original Japanese Run: 4–5 Minutes on a T4)

### LoRA versus QLoRA

- **LoRA**: Freeze the base-model weights and train only small matrices (adapters) added to some layers.
- **QLoRA**: Keep base-model weights quantized to 4-bit and train only adapters in the same way. The adapters themselves are not quantized.

QLoRA reduces the memory used to hold base-model weights. It uses about a quarter of the memory of 16-bit, so models too large to fit on a GPU otherwise can be trained (for 7 billion parameters, about 14GB at 16-bit versus about 3.5GB at 4-bit).

| Aspect | LoRA (no quantization) | QLoRA (4-bit) |
| --- | --- | --- |
| Weight memory | Large | Small (about 1/4 of 16-bit) |
| Base-model outputs | Unchanged | May change due to quantization error; the size of the effect depends on the model |
| Training speed | Baseline | Not faster. Often slower because 4-bit weights must be converted back for each computation |
| Dependencies | peft | peft plus bitsandbytes (requires a CUDA GPU) |

Notes for implementing QLoRA:

- Pass `BitsAndBytesConfig(load_in_4bit=True, ...)` when loading, then run `prepare_model_for_kbit_training(model)` before adding LoRA.
- Storage precision (4-bit) and compute precision (`bnb_4bit_compute_dtype`, such as 16-bit) are separate settings.
- Compare before and after training using the same quantization conditions for the base model. Measuring without quantization before training and with 4-bit afterward mixes quantization and training effects.
- Adapters are created for the base model used in training. Record the base-model ID, revision, and quantization settings together when distributing or reusing them.
- 4-bit training is discouraged for some models. One [training-library guide](https://unsloth.ai/docs/models/qwen3.5/fine-tune) discourages 4-bit training of Qwen3.5 because its quantization error is larger than usual.

This exercise's model fits on a T4 without quantization, so we train with LoRA. Consider LoRA first if the model fits; if it does not, consider QLoRA or a smaller model. You can examine memory and output differences when loading in 4-bit in section 8 (Advanced).

### Run Training

**What to Do**: Convert the 14 train examples to chat format (system / user / assistant), attach LoRA, and train for 20 epochs (40 steps). Save the trained adapter in `adapter/`.

- `target_modules` specifies layers where LoRA is inserted. We target attention projections (such as `q_proj`) and MLP projections (such as `gate_proj`). This model uses linear attention in 18 of its 24 layers; only the remaining 6 have `q_proj` and similar projections. All layers have MLPs.
- Batch 4 × gradient accumulation 2 = 8 examples per step. With 14 train examples, each epoch has 2 steps (8 and 6 examples), giving 40 steps over 20 epochs. The model sees each training example exactly 20 times. Learning-rate warmup and cosine decay apply over these 40 steps.
- The trained model is stored in `tuned`. Since LoRA is inserted into the base-model layers, generation through the original variable `model` also includes adapters. Temporarily disable adapters as in section 6 when you need before-training outputs.

**What to Look For**

- `trainable params: ... || all params: ... || trainable%: ...`: a feature of LoRA is that only a few percent of all parameters are trained.
- The `Training Loss` table (every 10 steps): in the original Japanese run, loss started around 1.5 and dropped to 0.01–0.02 by step 40. The translated data may give different values. If training ends without a decrease, check whether you changed the settings.
- `Training time: ... seconds | Steps: 40 | Epochs: 20.0`: the original Japanese run took 240–290 seconds on a T4; timing may differ with the translated data. Steps and epochs confirm that 14 training examples × 20 passes were run as calculated above.

**Harmless Warning**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` means end-token settings were aligned with the tokenizer.

**Troubleshooting**: If training stops midway or cannot run, read 4b below.

In [7]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # Training data pairs inputs (system + user) with desired outputs (assistant). Use the same build_messages as for evaluation
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 uses linear attention in 18 of 24 layers; only the remaining 6 have q/k/v/o_proj. All layers have gate/up/down_proj (MLP)
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # Normally leave unchanged. True enables 16-bit mixed-precision computation (CUDA only; weights stay 32-bit). Faster, but loss may become nan
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # Fraction of parameters trained with LoRA
t0 = time.time()
trainer.train()  # See 4b to resume if interrupted
print(f"Training time: {time.time()-t0:.0f} seconds | Steps: {trainer.state.global_step} | Epochs: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # Training mode enables dropout and makes evaluation outputs vary (assignment suppresses displaying the whole model)

Tokenizing train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/14 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/7 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,1.522592
20,0.056936
30,0.019628
40,0.016341


Training time: 158 seconds | Steps: 40 | Epochs: 20.0


### 4b. If Training Stops or Cannot Run

**Stopped midway (runtime disconnected, etc.)**: Checkpoints are saved every 10 steps in `/content/library_lora_lab/ckpt/`. If the files remain, rerun sections 0–3, then change `trainer.train()` in the training cell above to `trainer.train(resume_from_checkpoint=True)` and run it. Training resumes from the last checkpoint up to 40 steps (20 epochs). If the runtime and its files are gone, restart training or use the alternative below.

**Could not train (no GPU available or not enough time)**: Compare and discuss the instructor's results (outputs and scores after training) from the same notebook run on a T4. Cells from section 5 onward require the trained model `tuned`, so read the explanations alongside the instructor's results. When submitting, set `executed_training_myself` in section 7 to `False` and state clearly how far you ran the notebook yourself.

## 5. Evaluate After Training (Same test and Generation Settings as Before)

**What to Do**: Evaluate the trained model `tuned` on exactly the same 21 questions and generation settings, and compare before and after. The only change is whether adapters are present.

**What to Look For**

- Comparison table: how did `format_rate` (format compliance rate) and `content_rate` (content accuracy) change from before training?
- `Regressions`: questions that passed both format and content before training but failed afterward. Ideally the list is empty; inspect any regressions in your own run.
- Per-question table: read the remaining failures and identify their types. The original Japanese run included these types:
  - Applying a different FAQ item and making an assertion for a question without evidence (expected `needs_review` is `True`).
  - Evading a question with evidence (expected value is `False`) by requesting confirmation and saying "It is not listed".

In the original Japanese run, after-training answers tended to reproduce the training text (the original FAQ wording) almost verbatim. Check whether your outputs show the same pattern. Format and wording become consistent, but the model has not necessarily learned to judge "whether the FAQ contains evidence".

GPU training is not perfectly reproducible even with a fixed seed. Format compliance and content accuracy may be nearly the same while the remaining questions differ between runs. It is normal for your "remaining questions" to differ from your classmate's; compare failure types.

In [8]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# Questions that passed both format and content before training but failed afterward (regressions)
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("Regressions:", regressed)

# Original per-question outputs. Visually check format (JSON, required keys, types) and content (matching needs_review and required keywords) judgments
detail = pd.DataFrame([{"Question": b["question"], "Expected needs_review": b["gold_needs_review"],
                        "Before-training output": b["raw"], "Before format/content": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "After-training output": a["raw"], "After format/content": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

{"label": "after", "n": 21, "format_rate": 1.0, "content_rate": 0.9523809523809523, "both_rate": 0.9523809523809523, "seconds": 35.8}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,21,1.0,0.428571,0.428571,43.4
after,21,1.0,0.952381,0.952381,35.8


Regressions: []


,Question,Expected needs_review,Before-training output,Before format/content,After-training output,After format/content
0,When must I return a book after borrowing it?,False,"{""answer"": ""Return borrowed books at the front desk."", ""needs_review"": true}",Pass/Fail,"{""answer"": ""Return borrowed books at the front desk."", ""needs_review"": false}",Pass/Fail
1,How many days does a book loan last?,False,"{""answer"": ""14"", ""needs_review"": false}",Pass/Pass,"{""answer"": ""Books may be borrowed for 14 days."", ""needs_review"": false}",Pass/Pass
2,What is the permitted borrowing duration for a book?,False,"{""answer"": ""14 days"", ""needs_review"": false}",Pass/Pass,"{""answer"": ""Books may be borrowed for 14 days."", ""needs_review"": false}",Pass/Pass
3,How many books may a member have on loan simultaneously?,False,"{""answer"": ""3"", ""needs_review"": false}",Pass/Pass,"{""answer"": ""Each member may borrow up to 3 books at a time."", ""needs_review"": false}",Pass/Pass
4,What is the simultaneous book-loan limit?,False,"{""answer"": ""3"", ""needs_review"": false}",Pass/Pass,"{""answer"": ""Each member may borrow up to 3 books at a time."", ""needs_review"": false}",Pass/Pass
5,What is the largest number of books a member can check out at once?,False,"{""answer"": ""3"", ""needs_review"": false}",Pass/Pass,"{""answer"": ""Each member may borrow up to 3 books at a time."", ""needs_review"": false}",Pass/Pass
6,Where can I hand back a borrowed book?,False,"{""answer"": ""The front desk"", ""needs_review"": true}",Pass/Fail,"{""answer"": ""Return borrowed books at the front desk."", ""needs_review"": false}",Pass/Pass
7,Which desk accepts book returns?,False,"{""answer"": ""Front desk"", ""needs_review"": false}",Pass/Fail,"{""answer"": ""Return borrowed books at the front desk."", ""needs_review"": false}",Pass/Pass
8,Where should a member bring books at the end of a loan?,False,"{""answer"": ""Return borrowed books at the front desk."", ""needs_review"": true}",Pass/Fail,"{""answer"": ""Return borrowed books at the front desk."", ""needs_review"": false}",Pass/Pass
9,On which days and at what times can I visit the library?,False,"{""answer"": ""Monday to Friday from 09:00 to 17:00"", ""needs_review"": false}",Pass/Pass,"{""answer"": ""The library is open Monday to Friday from 09:00 to 17:00."", ""needs_review"": false}",Pass/Pass


## 6. Knowledge Questions (Learning a Format Does Not Guarantee Knowledge)

**What to Do**: Separately from the format experiment, ask 3 factual questions absent from both the FAQ and training data (plus one FAQ-covered question as a control). The desired response is "Confirmation is needed (`needs_review: true`)".

Classify responses into three categories.

| Classification | Meaning |
| --- | --- |
| `format_error` | Does not follow the JSON format (common before training in the original Japanese run) |
| `needs_review` | Returns "It is not listed. Confirmation is needed" (the desired response) |
| `asserted` | Makes an assertion with `needs_review: false`. For a question absent from the FAQ, this is an unsupported assertion (fabrication) |

**What to Look For**: In the original Japanese run, format errors were common before training, and both "confirmation needed" and "assertion" responses appeared after training. Check the response types in your own run. The table also shows original outputs. For assertions, read what the model seems to rely on (whether it reuses a different FAQ item). Report results for this dataset rather than deciding in advance that "fine-tuning can never correct knowledge".

Before-training outputs are obtained by temporarily disabling adapters (`tuned.disable_adapter()`).

In [9]:
KNOWLEDGE_QUESTIONS = [
    "What is the library director's name?",
    "Does the library offer home delivery?",
    "What happens if a borrowed book is damaged?",
    "How long can I borrow a book?",  # FAQ-covered control
]

def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

grounded_questions = {ex["question"] for ex in train + valid + test if not ex["needs_review"]}
knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA is inserted into base-model layers, so generation with `model` also uses adapters after training. Temporarily disable adapters for before-training outputs
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"Question": q, "In FAQ": q in grounded_questions, "Before-training classification": classify(before_raw), "Before-training output": before_raw,
                 "After-training classification": classify(after_raw), "After-training output": after_raw})
display(pd.DataFrame(knowledge_rows))

,Question,In FAQ,Before-training classification,Before-training output,After-training classification,After-training output
0,What is the library director's name?,False,needs_review,"{""answer"": ""Confirmation is needed"", ""needs_review"": true}",needs_review,"{""answer"": ""This FAQ does not contain that information. Confirmation from the library staff is needed."", ""needs_review"": true}"
1,Does the library offer home delivery?,False,needs_review,"{""answer"": ""No"", ""needs_review"": true}",needs_review,"{""answer"": ""This FAQ does not contain that information. Confirmation from the library staff is needed."", ""needs_review"": true}"
2,What happens if a borrowed book is damaged?,False,needs_review,"{""answer"": ""Confirmation is needed"", ""needs_review"": true}",needs_review,"{""answer"": ""This FAQ does not contain that information. Confirmation from the library staff is needed."", ""needs_review"": true}"
3,How long can I borrow a book?,True,asserted,"{""answer"": ""14 days"", ""needs_review"": false}",asserted,"{""answer"": ""Books may be borrowed for 14 days."", ""needs_review"": false}"


## 7. Comparison Table and Submission

**What to Do**: Combine settings, library versions, before/after summaries, and regressed questions into one `result.json`.

**What to Submit**

- `result.json`, `eval_before.json`, and `eval_after.json` (in `/content/library_lora_lab`; right-click in the file list on the left → **Download**. With `USE_DRIVE = True`, they are also copied to Drive)
- The comparison table from section 5 and an explanation of remaining failures (failure types)
- Results from section 6 (responses that changed to confirmation requests and those that remained assertions)
- How far you ran training yourself. If you used the instructor's results, set `executed_training_myself` to `False`.

In [10]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (no quantization)",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # Set False if using the instructor's results
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Saved to Drive:", CONFIG["drive_dir"])

{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA (no quantization)",
  "before": {
    "label": "before",
    "n": 21,
    "format_rate": 1.0,
    "content_rate": 0.42857142857142855,
    "both_rate": 0.42857142857142855,
    "seconds": 43.4
  },
  "after": {
    "label": "after",
    "n": 21,
    "format_rate": 1.0,
    "content_rate": 0.9523809523809523,
    "both_rate": 0.9523809523809523,
    "seconds": 35.8
  },
  "regressed": [],
  "executed_training_myself": true
}


## 8. Advanced: Load with 4-bit Quantization and Compare Memory and Outputs (Optional)

**What to Do**: Load the same model in 4-bit (NF4) and observe changes in weight memory and before-training outputs. QLoRA adds LoRA to this 4-bit base model and trains it. No training is done here.

- bitsandbytes requires a CUDA GPU, so this runs only in a GPU runtime. Installation takes about 1 minute.
- 4-bit loading places the model on the GPU with `device_map="auto"` (matching bitsandbytes' loading method).
- If you also try training the 4-bit model, run `prepare_model_for_kbit_training(model_4bit)` first, follow section 4's training procedure, and evaluate both before and after using a 4-bit base model.

**What to Look For**

- Weight memory: in the original Japanese run, 4-bit used less than 1GB, versus about 3GB at 32-bit. Compare the displayed values in your run.
- Comparison and per-question tables: since no training occurred, differences from the 32-bit before-training results (`before`) reflect quantization. In the original Japanese run, some questions improved, while other outputs had `needs_review` as the string `"false"` and failed type checks, or asserted "yes" or "none" for questions without evidence. Check the kinds and direction of changes in your run.

In [11]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # Storage precision: 4-bit (NF4)
                         bnb_4bit_compute_dtype=torch.float16,             # Compute precision: 16-bit (T4 does not support bf16)
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("Weight memory (GB) | No quantization (32-bit):", BASE_MEMORY_GB, "| 4-bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# No training occurred, so differences from the unquantized before-training results (before) reflect quantization
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"Question": r["question"], "Expected needs_review": r["gold_needs_review"], "4-bit before-training output": r["raw"],
                       "Format/content": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "Format failure reason": r["format_reason"]} for r in q_rows]))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Weight memory (GB) | No quantization (32-bit): 3.01 | 4-bit: 0.76
{"label": "before_4bit", "n": 21, "format_rate": 1.0, "content_rate": 0.14285714285714285, "both_rate": 0.14285714285714285, "seconds": 28.2}


,n,format_rate,content_rate,both_rate,seconds
label,,,,,
before,21,1.0,0.428571,0.428571,43.4
before_4bit,21,1.0,0.142857,0.142857,28.2


,Question,Expected needs_review,4-bit before-training output,Format/content,Format failure reason
0,When must I return a book after borrowing it?,False,"{""answer"": ""Return borrowed books at the front desk."", ""needs_review"": true}",Pass/Fail,
1,How many days does a book loan last?,False,"{""answer"": ""14"", ""needs_review"": true}",Pass/Fail,
2,What is the permitted borrowing duration for a book?,False,"{""answer"": ""14 days"", ""needs_review"": false}",Pass/Pass,
3,How many books may a member have on loan simultaneously?,False,"{""answer"": ""3"", ""needs_review"": false}",Pass/Pass,
4,What is the simultaneous book-loan limit?,False,"{""answer"": ""3"", ""needs_review"": true}",Pass/Fail,
5,What is the largest number of books a member can check out at once?,False,"{""answer"": ""3"", ""needs_review"": false}",Pass/Pass,
6,Where can I hand back a borrowed book?,False,"{""answer"": ""front desk"", ""needs_review"": true}",Pass/Fail,
7,Which desk accepts book returns?,False,"{""answer"": ""front desk"", ""needs_review"": true}",Pass/Fail,
8,Where should a member bring books at the end of a loan?,False,"{""answer"": ""front desk"", ""needs_review"": true}",Pass/Fail,
9,On which days and at what times can I visit the library?,False,"{""answer"": ""Monday to Friday from 09:00 to 17:00"", ""needs_review"": true}",Pass/Fail,
